In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 문장 -> 벡터(1차원 숫자 배열 [8.1,9.1, 2, 5, 4, 3....])
- openAi API : https://platform.openai.com/의 키(OPENAI_API_KEY)를 .env등록
- pstage : https://console.upstage.ai/의 키(UPSTAGE_API_KEY)를 .env등록

## 1. 환경변수 load

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## 2. 유사도 계산하는 방법 : https://www.pinecone.io/learn/vector-similarity
    1. 유클리드 거리 : 두 벡터간의 거리가 가까운지
    2. 코사인유사도 : 두 벡터간 방향이 유사한지
    3. dot product : 두 벡터간의 곱을 사용하여 거리와 방향을 모두 고려

In [3]:
import numpy as np
def cosine_similarity(vec1, vec2):
    """두 백터 사이의 코사인 유사도 계산"""
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1) # 벡터의 길이
    norm_vec2 = np.linalg.norm(vec2)
    if norm_vec1==0 or norm_vec2==0:
        return 0.0
    return dot_product / (norm_vec1*norm_vec2)

## 3. openAI API의 embedding model 사용

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
# text-embedding-3-large
response = openai_client.embeddings.create(
    input="king",
    model="text-embedding-3-large"
)

In [6]:
import numpy as np
king_vector = np.array(response.data[0].embedding)
print(king_vector.shape)
print(king_vector)

(3072,)
[ 0.01040417  0.02499519 -0.0014776  ...  0.00835009  0.01049861
 -0.00254005]


In [7]:
queen_response = openai_client.embeddings.create(
    input="queen",
    model="text-embedding-3-large"
)

In [8]:
queen_vector = np.array(queen_response.data[0].embedding)
print(queen_vector)
print(queen_vector.shape)

[-0.01385735  0.0008602  -0.0167823  ...  0.00017693  0.01159847
  0.00638929]
(3072,)


In [9]:
king_queen_similarity = cosine_similarity(king_vector, queen_vector)
print('king과 queen의 유사도 :',king_queen_similarity)

king과 queen의 유사도 : 0.5552268369726675


In [10]:
slave_response = openai_client.embeddings.create(
    input="slave",
    model="text-embedding-3-large"
)
slave_vector = np.array(slave_response.data[0].embedding)
print(slave_vector.shape)
print(slave_vector)

(3072,)
[-0.02002587  0.00621112  0.01189955 ...  0.00094484 -0.02675561
 -0.00584769]


In [11]:
king_slave_similarity = cosine_similarity(king_vector, slave_vector)
print('king과 slave유사도 :', king_slave_similarity)

king과 slave유사도 : 0.294827248041228


In [12]:
# 한국어 문장을 벡터로 바꿔도 유사도는 비슷해야 할 듯

In [13]:
kor_king_response = openai_client.embeddings.create(
    input="왕",
    model="text-embedding-3-large"
)

In [14]:
kor_king_vector = np.array(kor_king_response.data[0].embedding)
print(kor_king_vector.shape)

(3072,)


In [15]:
kor_queen_response = openai_client.embeddings.create(
    input="여왕",
    model="text-embedding-3-large"
)
kor_queen_vector = np.array(kor_queen_response.data[0].embedding)
print(kor_queen_vector.shape)

(3072,)


In [16]:
# 왕과 여왕의 유사
cosine_similarity(kor_king_vector, kor_queen_vector)

0.48733449549538954

In [17]:
kor_slave_response = openai_client.embeddings.create(
    input="거지",
    model="text-embedding-3-large"
)
kor_slave_vector = np.array(kor_slave_response.data[0].embedding)
print(kor_slave_vector.shape)

(3072,)


In [18]:
# 왕과 거지의 유사
cosine_similarity(kor_king_vector, kor_slave_vector)

0.2552452064791607

In [19]:
# king과 왕의 유사도
cosine_similarity(king_vector, kor_king_vector)

0.5474873912140233

## 4. upstage의 embedding model 사용
- 한국에 embedding에는 openai보다 성능이 훨씬 좋다

In [20]:
import os
upstage_api_key = os.getenv("UPSTAGE_API_KEY")
upstage_client = OpenAI(
    api_key=upstage_api_key,
    base_url="https://api.upstage.ai/v1"
)

In [21]:
up_king_response = upstage_client.embeddings.create(
    input="king",
    model="embedding-query"
)

In [22]:
up_king_vector = np.array(up_king_response.data[0].embedding)
print(up_king_vector.shape)
print(up_king_vector)

(4096,)
[-0.01187134 -0.02058411 -0.00674438 ... -0.01082611  0.00244713
  0.01517487]


In [23]:
up_queen_response = upstage_client.embeddings.create(
    input="queen",
    model="embedding-query"
)
up_queen_vector = np.array(up_queen_response.data[0].embedding)
print(up_queen_vector.shape)

(4096,)


In [24]:
cosine_similarity(up_king_vector, up_queen_vector)

0.6277983746920601

In [25]:
up_kor_king_response = upstage_client.embeddings.create(
    input="왕",
    model="embedding-query"
)
up_kor_king_vector = np.array(up_kor_king_response.data[0].embedding)
print(up_kor_king_vector.shape)

(4096,)


In [26]:
# king과 왕의 유사도
cosine_similarity(up_king_vector, up_kor_king_vector)

0.852149171074866